# 28: Explore the Data, Build a Baseline

## Your First Day as a Junior ML Engineer

Your team lead walks over: *"We need a spam classifier for our SMS gateway. Here's the data. Can you explore it and get a quick baseline running by end of day?"*

This is literally how real ML projects start. No clean toy datasets. No predefined train/test splits. Just a CSV and a deadline.

**What you'll build today:**
- A complete data exploration (EDA) pipeline
- 3 baseline models, each better than the last
- A proper evaluation that you can present to your team

## What You'll Learn
- [ ] Explore a real dataset and identify data quality issues
- [ ] Handle class imbalance, duplicates, and text preprocessing
- [ ] Build 3 progressively better baselines (rules → features → TF-IDF)
- [ ] Evaluate models with metrics that matter for imbalanced data

## Connection to Previous Lessons

| What you learned | How it connects here |
|-----------------|---------------------|
| **Lesson 2**: Features, labels, train/test split | Now on real data with real problems (imbalance, duplicates) |
| **Lesson 5**: Bag-of-words, text preprocessing | Same techniques, but messier text (emojis, abbreviations, URLs) |
| **Lesson 6**: sklearn pipelines | We'll build real sklearn pipelines for the baselines |
| **Lesson 13**: Word embeddings, TF-IDF | TF-IDF becomes our strongest baseline feature |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

# Load the data
df = pd.read_csv('../../data/sms_spam.csv')

print(f"Dataset loaded: {len(df)} messages")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst 5 rows:")
df.head()

## 1. Explore: What Are We Working With?

Before writing any model code, a good ML engineer spends time **understanding the data**.
This step catches problems that would silently break your model later.

In [ ]:
# --- Class Distribution ---
print("Class distribution:")
print(df['label'].value_counts())
print(f"\nSpam ratio: {(df['label'] == 'spam').mean():.1%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'salmon'])
axes[0].set_title('Message Counts by Class')
axes[0].set_ylabel('Count')

# Percentage
df['label'].value_counts(normalize=True).plot(kind='pie', ax=axes[1], 
    autopct='%1.1f%%', colors=['steelblue', 'salmon'])
axes[1].set_title('Class Proportions')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print("\n⚠️  13.4% spam = IMBALANCED dataset")
print("   A model that always predicts 'ham' gets 86.6% accuracy!")
print("   We need metrics beyond accuracy.")

In [ ]:
# --- Data Quality ---
print("=== Data Quality Report ===\n")

# Missing values
print(f"Missing values: {df.isnull().sum().sum()}")

# Duplicates
n_dupes = df.duplicated(subset='message').sum()
print(f"Duplicate messages: {n_dupes}")

# Show some duplicates
if n_dupes > 0:
    dupe_examples = df[df.duplicated(subset='message', keep=False)].sort_values('message').head(4)
    print(f"\nExample duplicates:")
    for _, row in dupe_examples.iterrows():
        print(f"  [{row['label']}] {row['message'][:60]}...")

# Message lengths
df['msg_length'] = df['message'].str.len()
print(f"\nMessage length stats:")
print(f"  Min: {df['msg_length'].min()} chars")
print(f"  Max: {df['msg_length'].max()} chars")
print(f"  Mean: {df['msg_length'].mean():.0f} chars")

print(f"\n✓ Data quality check complete")

In [ ]:
# --- Message Length by Class ---
fig, ax = plt.subplots(figsize=(10, 5))

for label, color in [('ham', 'steelblue'), ('spam', 'salmon')]:
    subset = df[df['label'] == label]
    ax.hist(subset['msg_length'], bins=50, alpha=0.6, label=label, color=color)

ax.set_xlabel('Message Length (characters)')
ax.set_ylabel('Count')
ax.set_title('Message Length Distribution by Class')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Average ham length:  {df[df['label']=='ham']['msg_length'].mean():.0f} chars")
print(f"Average spam length: {df[df['label']=='spam']['msg_length'].mean():.0f} chars")
print("\n💡 Spam messages tend to be LONGER — this alone is a useful feature!")

In [ ]:
# --- Most Common Words per Class ---
from collections import Counter

def get_top_words(texts, n=15):
    words = ' '.join(texts).lower().split()
    # Remove very short words
    words = [w for w in words if len(w) > 2]
    return Counter(words).most_common(n)

ham_words = get_top_words(df[df['label']=='ham']['message'])
spam_words = get_top_words(df[df['label']=='spam']['message'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

words, counts = zip(*ham_words)
axes[0].barh(words, counts, color='steelblue')
axes[0].set_title('Top Words in Ham')
axes[0].invert_yaxis()

words, counts = zip(*spam_words)
axes[1].barh(words, counts, color='salmon')
axes[1].set_title('Top Words in Spam')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print("💡 Notice: spam has 'free', 'call', 'txt', 'claim' — action/urgency words")
print("   Ham has 'you', 'the', 'have' — normal conversation words")

## 2. Clean: Fix the Problems

Now we know what's wrong. Let's fix it:
1. Remove duplicates (keep first occurrence)
2. Create a binary label column (0 = ham, 1 = spam)
3. Basic text preprocessing

In [ ]:
# --- Clean the Data ---
print(f"Before cleaning: {len(df)} messages")

# Remove duplicates
df_clean = df.drop_duplicates(subset='message', keep='first').copy()
print(f"After removing duplicates: {len(df_clean)} messages (-{len(df) - len(df_clean)})")

# Binary labels
df_clean['is_spam'] = (df_clean['label'] == 'spam').astype(int)

# Basic text preprocessing
def preprocess_text(text):
    """Clean text for ML: lowercase, remove punctuation, normalize whitespace."""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', ' URL ', text)  # Replace URLs
    text = re.sub(r'\d+', ' NUM ', text)                # Replace numbers
    text = re.sub(r'[^\w\s]', ' ', text)                # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()             # Normalize whitespace
    return text

df_clean['clean_text'] = df_clean['message'].apply(preprocess_text)

# Show examples
print("\nPreprocessing examples:")
for _, row in df_clean.head(3).iterrows():
    print(f"  Original: {row['message'][:60]}")
    print(f"  Cleaned:  {row['clean_text'][:60]}")
    print()

print(f"✓ Data cleaned and preprocessed")

In [ ]:
from sklearn.model_selection import train_test_split

# Stratified split: preserve class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    df_clean['clean_text'], 
    df_clean['is_spam'],
    test_size=0.2,
    random_state=42,
    stratify=df_clean['is_spam']  # Important for imbalanced data!
)

print(f"Train set: {len(X_train)} messages")
print(f"Test set:  {len(X_test)} messages")
print(f"\nSpam ratio - Train: {y_train.mean():.1%}, Test: {y_test.mean():.1%}")
print("✓ Ratios match — stratified split worked!")

## 3. Baseline 1: Keyword Rules

The simplest possible approach. No ML at all.
*"If the message contains certain spam keywords, flag it."*

Every ML project should start with a dumb baseline — it tells you how much value ML actually adds.

In [ ]:
# --- Baseline 1: Keyword Rules ---
SPAM_KEYWORDS = ['free', 'win', 'winner', 'prize', 'claim', 'urgent', 
                 'congratulations', 'cash', 'offer', 'click', 'subscribe']

def rule_based_predict(text):
    """Predict spam if any keyword is found."""
    text_lower = text.lower()
    return int(any(keyword in text_lower for keyword in SPAM_KEYWORDS))

# Predict on test set
y_pred_rules = X_test.apply(rule_based_predict)

# Evaluate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(y_true, y_pred, name="Model"):
    """Print key metrics for a binary classifier."""
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    print(f"=== {name} ===")
    print(f"  Accuracy:  {acc:.3f}")
    print(f"  Precision: {prec:.3f}  (of predicted spam, how many are actually spam?)")
    print(f"  Recall:    {rec:.3f}  (of actual spam, how many did we catch?)")
    print(f"  F1 Score:  {f1:.3f}  (harmonic mean of precision & recall)")
    print()
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

results_rules = evaluate_model(y_test, y_pred_rules, "Baseline 1: Keyword Rules")
print("1/3 baselines done")

print("💡 High precision but LOW recall — we catch few spam messages")
print("   Many spam messages don't contain our keywords")

## 4. Baseline 2: Hand-Crafted Features + Logistic Regression

Instead of keyword matching, extract **features** that describe each message numerically, then let a model learn the patterns.

In [ ]:
# --- Baseline 2: Hand-Crafted Features ---
def extract_features(texts):
    """Extract 8 numerical features from text messages."""
    features = pd.DataFrame()
    
    features['length'] = texts.str.len()
    features['word_count'] = texts.str.split().str.len()
    features['caps_ratio'] = texts.apply(lambda x: sum(1 for c in x if c.isupper()) / (len(x) + 1))
    features['digit_ratio'] = texts.apply(lambda x: sum(1 for c in x if c.isdigit()) / (len(x) + 1))
    features['has_url'] = texts.str.contains(r'http|www|URL', case=False).astype(int)
    features['exclamation_count'] = texts.str.count('!')
    features['special_char_count'] = texts.str.count(r'[£$€¥]')
    features['avg_word_length'] = texts.apply(lambda x: np.mean([len(w) for w in x.split()]) if x.split() else 0)
    
    return features

# Use ORIGINAL text for features (before preprocessing removed punctuation)
X_train_feat = extract_features(df_clean.loc[X_train.index, 'message'])
X_test_feat = extract_features(df_clean.loc[X_test.index, 'message'])

print("Features extracted:")
print(X_train_feat.describe().round(2))

In [ ]:
# --- Which features separate spam from ham? ---
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

feature_names = X_train_feat.columns
for i, (feat, ax) in enumerate(zip(feature_names, axes.flat)):
    for label, color in [(0, 'steelblue'), (1, 'salmon')]:
        data = X_train_feat[y_train == label][feat]
        ax.hist(data, bins=30, alpha=0.6, color=color, label='ham' if label == 0 else 'spam', density=True)
    ax.set_title(feat, fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions: Ham vs Spam', fontsize=14)
plt.tight_layout()
plt.show()

print("💡 Best separators: length, caps_ratio, has_url, special_char_count")
print("   Spam messages are longer, LOUDER, and contain more URLs/symbols")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Scale features (important for logistic regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_feat)
X_test_scaled = scaler.transform(X_test_feat)

# Train
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

# Predict
y_pred_features = lr_model.predict(X_test_scaled)
results_features = evaluate_model(y_test, y_pred_features, "Baseline 2: Features + LogReg")
print("2/3 baselines done")

# Feature importance
print("Feature importance (coefficient magnitude):")
for name, coef in sorted(zip(feature_names, np.abs(lr_model.coef_[0])), key=lambda x: -x[1]):
    bar = '█' * int(coef * 5)
    print(f"  {name:20s} {bar} ({coef:.2f})")

## 5. Baseline 3: TF-IDF + Logistic Regression

Instead of hand-crafting features, let the algorithm learn from **word frequencies** directly. TF-IDF (Term Frequency - Inverse Document Frequency) weighs words by how important they are to each document.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

# Build a pipeline: TF-IDF → Logistic Regression
tfidf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ('clf', LogisticRegression(random_state=42, max_iter=1000))
])

# Train
tfidf_pipeline.fit(X_train, y_train)

# Predict
y_pred_tfidf = tfidf_pipeline.predict(X_test)
results_tfidf = evaluate_model(y_test, y_pred_tfidf, "Baseline 3: TF-IDF + LogReg")
print("3/3 baselines done")

# Top features
feature_names_tfidf = tfidf_pipeline['tfidf'].get_feature_names_out()
coefs = tfidf_pipeline['clf'].coef_[0]
top_spam = sorted(zip(feature_names_tfidf, coefs), key=lambda x: -x[1])[:10]
top_ham = sorted(zip(feature_names_tfidf, coefs), key=lambda x: x[1])[:10]

print("Top spam indicators:")
for word, coef in top_spam:
    print(f"  '{word}': {coef:.2f}")
print("\nTop ham indicators:")
for word, coef in top_ham:
    print(f"  '{word}': {coef:.2f}")

In [ ]:
# --- Final Comparison ---
import pandas as pd

comparison = pd.DataFrame({
    'Keyword Rules': results_rules,
    'Features + LogReg': results_features,
    'TF-IDF + LogReg': results_tfidf,
}).T

print("=== Baseline Comparison ===")
print(comparison.round(3).to_string())

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
comparison[['precision', 'recall', 'f1']].plot(kind='bar', ax=ax, colormap='Set2')
ax.set_title('Baseline Comparison')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("\n💡 TF-IDF wins by a large margin!")
print("   But notice: even TF-IDF misses some spam (recall < 1.0)")
print("   Can we do better? → Lesson 29: Fine-tune a Transformer!")

## What Can Go Wrong: Data Leakage

**Data leakage** = information from the test set accidentally leaks into training.

The most common form: preprocessing or feature engineering that uses the FULL dataset before splitting.

Example: If you compute TF-IDF on ALL data (train + test) before splitting, the model "sees" test vocabulary during training. Metrics look great in the notebook, but the model fails on truly unseen data.

**Rule:** Always split FIRST, then preprocess/featurize train and test SEPARATELY.

In [ ]:
# --- Data Leakage Demo ---
# WRONG: fit TF-IDF on all data before splitting
tfidf_leaky = TfidfVectorizer(max_features=5000)
X_all_tfidf = tfidf_leaky.fit_transform(df_clean['clean_text'])  # Leaky! Uses test data

# Split AFTER fitting
X_train_leak = X_all_tfidf[:len(X_train)]
X_test_leak = X_all_tfidf[len(X_train):]

leak_model = LogisticRegression(random_state=42, max_iter=1000)
leak_model.fit(X_train_leak, y_train)
leak_acc = leak_model.score(X_test_leak, y_test)

# CORRECT: fit TF-IDF only on train
tfidf_clean = TfidfVectorizer(max_features=5000)
X_train_clean = tfidf_clean.fit_transform(X_train)
X_test_clean = tfidf_clean.transform(X_test)  # Only transform, don't fit!

clean_model = LogisticRegression(random_state=42, max_iter=1000)
clean_model.fit(X_train_clean, y_train)
clean_acc = clean_model.score(X_test_clean, y_test)

print(f"With data leakage:    {leak_acc:.4f}")
print(f"Without data leakage: {clean_acc:.4f}")
print(f"Difference:           {leak_acc - clean_acc:+.4f}")
print()
print("⚠️  The difference may seem small here, but on larger datasets")
print("   or with more complex features, leakage can inflate accuracy by 5-10%!")
print("   Your model looks great in the notebook but fails in production.")

## Exercises

Build your own features and test your understanding of the pipeline.
Each exercise has built-in validation — run the cell and check if you pass!

In [ ]:
# =================================================================
# Exercise 1: Build Your Own Feature
# =================================================================
# 
# Spam often uses URGENCY tactics: "Act now!", "Limited time!", "Hurry!"
# Create a feature that counts urgency words in a message.
#
# Steps:
#   1. Define a list of urgency words
#   2. Count how many appear in each message
#   3. Test that spam messages have higher urgency scores
#
# Hints:
#   - Convert text to lowercase before checking
#   - Use: sum(1 for word in urgency_words if word in text.lower())
# =================================================================

# YOUR CODE HERE:
urgency_words = None  # List of 5+ urgency words (e.g., "urgent", "now", "hurry", ...)

def count_urgency(text):
    """Count urgency words in a text message. Return an integer."""
    # YOUR CODE HERE:
    return None

# =================================================================
# TESTS — run this cell to check your work
# =================================================================
# Test 1: urgency_words should be a list with at least 5 words
assert urgency_words is not None, "Define your urgency_words list!"
assert isinstance(urgency_words, list), "urgency_words should be a list"
assert len(urgency_words) >= 5, f"Need at least 5 urgency words, got {len(urgency_words)}"
print(f"✓ Test 1 passed: {len(urgency_words)} urgency words defined")

# Test 2: count_urgency should return an integer
assert count_urgency is not None, "Implement count_urgency!"
test_result = count_urgency("URGENT! Act now! Limited time offer!")
assert isinstance(test_result, (int, np.integer)), f"Should return int, got {type(test_result)}"
assert test_result > 0, "Should find urgency words in 'URGENT! Act now! Limited time offer!'"
print(f"✓ Test 2 passed: found {test_result} urgency words in test message")

# Test 3: normal message should have 0 urgency
normal_result = count_urgency("Hey, are you coming to the party tonight?")
assert normal_result == 0, f"Normal message should have 0 urgency words, got {normal_result}"
print(f"✓ Test 3 passed: normal message has 0 urgency words")

# Test 4: spam should have higher urgency on average
train_msgs = df_clean.loc[X_train.index, 'message']
spam_urgency = train_msgs[y_train == 1].apply(count_urgency).mean()
ham_urgency = train_msgs[y_train == 0].apply(count_urgency).mean()
assert spam_urgency > ham_urgency, f"Spam urgency ({spam_urgency:.2f}) should be higher than ham ({ham_urgency:.2f})"
print(f"✓ Test 4 passed: spam avg urgency ({spam_urgency:.2f}) > ham avg ({ham_urgency:.2f})")

print(f"\n🎉 Exercise 1 complete! Your urgency feature separates spam from ham.")

In [ ]:
# =================================================================
# Exercise 2: Beat the TF-IDF Baseline
# =================================================================
#
# Combine your hand-crafted features WITH TF-IDF features.
# The idea: TF-IDF captures word patterns, your features capture
# structural patterns (length, urgency, URLs). Together they're stronger.
#
# Steps:
#   1. Create hand-crafted features for train and test
#   2. Create TF-IDF features for train and test
#   3. Concatenate them horizontally (np.hstack or scipy.sparse.hstack)
#   4. Train LogisticRegression on the combined features
#   5. Evaluate — does it beat TF-IDF alone?
#
# Hints:
#   - TF-IDF returns a sparse matrix. Use scipy.sparse.hstack to combine.
#   - from scipy.sparse import hstack as sparse_hstack
#   - sparse_hstack([tfidf_matrix, dense_features_as_sparse])
#   - Convert dense to sparse: from scipy.sparse import csr_matrix
# =================================================================

from scipy.sparse import hstack as sparse_hstack, csr_matrix

# Step 1: Hand-crafted features (reuse extract_features from above)
# Already computed: X_train_feat, X_test_feat

# Step 2: TF-IDF features
tfidf_ex = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf_ex = tfidf_ex.fit_transform(X_train)
X_test_tfidf_ex = tfidf_ex.transform(X_test)

# YOUR CODE HERE:
# Step 3: Combine features
X_train_combined = None  # sparse_hstack([tfidf_matrix, hand_crafted_matrix])
X_test_combined = None

# Step 4: Train model
combined_model = None  # LogisticRegression(...)

# Step 5: Predict
y_pred_combined = None

# =================================================================
# TESTS
# =================================================================
assert X_train_combined is not None, "Create X_train_combined by combining TF-IDF + hand-crafted features"
assert X_test_combined is not None, "Create X_test_combined the same way"
print(f"✓ Test 1: Combined features shape: {X_train_combined.shape}")
assert X_train_combined.shape[1] > X_train_tfidf_ex.shape[1], \
    f"Combined should have MORE features than TF-IDF alone ({X_train_combined.shape[1]} vs {X_train_tfidf_ex.shape[1]})"
print(f"✓ Test 2: More features than TF-IDF alone ({X_train_combined.shape[1]} > {X_train_tfidf_ex.shape[1]})")

assert combined_model is not None, "Train a LogisticRegression on X_train_combined"
assert y_pred_combined is not None, "Predict with your combined model"

results_combined = evaluate_model(y_test, y_pred_combined, "Your Model: TF-IDF + Features")

combined_f1 = f1_score(y_test, y_pred_combined)
tfidf_f1 = results_tfidf['f1']

if combined_f1 > tfidf_f1:
    print(f"🎉 You beat the TF-IDF baseline! ({combined_f1:.3f} > {tfidf_f1:.3f})")
else:
    print(f"Close! Your F1: {combined_f1:.3f}, TF-IDF: {tfidf_f1:.3f}")
    print("💡 Try adding more features or tuning TfidfVectorizer parameters")

print("\n✓ Exercise 2 complete!")

In [ ]:
# =================================================================
# Exercise 3: Error Analysis — Find Where Your Model Fails
# =================================================================
#
# A good ML engineer doesn't just measure accuracy — they look at
# the SPECIFIC messages the model gets wrong and ask WHY.
#
# Steps:
#   1. Get predicted probabilities from the TF-IDF model
#   2. Find the FALSE POSITIVES (ham predicted as spam) — these are
#      the worst errors (blocking legitimate messages!)
#   3. Find the FALSE NEGATIVES (spam predicted as ham) — missed spam
#   4. Print examples and identify patterns
#
# Hints:
#   - Use tfidf_pipeline.predict_proba(X_test)[:, 1] for spam probability
#   - False positives: (y_test == 0) & (y_pred_tfidf == 1)
#   - False negatives: (y_test == 1) & (y_pred_tfidf == 0)
# =================================================================

# YOUR CODE HERE:
# Get spam probabilities
spam_probs = None  # tfidf_pipeline.predict_proba(...)[:, 1]

# Find false positives (ham incorrectly flagged as spam)
false_positives = None  # Boolean mask

# Find false negatives (spam that slipped through)
false_negatives = None  # Boolean mask

# =================================================================
# TESTS
# =================================================================
assert spam_probs is not None, "Get probabilities with predict_proba!"
assert len(spam_probs) == len(y_test), f"Should have {len(y_test)} probabilities"
assert 0 <= spam_probs.min() and spam_probs.max() <= 1, "Probabilities must be between 0 and 1"
print(f"✓ Test 1: Got {len(spam_probs)} probability scores")

assert false_positives is not None, "Create the false_positives boolean mask"
assert false_negatives is not None, "Create the false_negatives boolean mask"

n_fp = false_positives.sum()
n_fn = false_negatives.sum()
print(f"✓ Test 2: Found {n_fp} false positives, {n_fn} false negatives")

# Show the errors
test_messages = df_clean.loc[X_test.index, 'message']

if n_fp > 0:
    print(f"\n--- FALSE POSITIVES (ham flagged as spam) — THE WORST ERRORS ---")
    fp_indices = X_test.index[false_positives]
    for idx in fp_indices[:5]:
        msg = df_clean.loc[idx, 'message']
        prob = spam_probs[X_test.index.get_loc(idx)]
        print(f"  [{prob:.0%} spam] {msg[:80]}...")
    print()

if n_fn > 0:
    print(f"--- FALSE NEGATIVES (spam that got through) ---")
    fn_indices = X_test.index[false_negatives]
    for idx in fn_indices[:5]:
        msg = df_clean.loc[idx, 'message']
        prob = spam_probs[X_test.index.get_loc(idx)]
        print(f"  [{prob:.0%} spam] {msg[:80]}...")

print(f"\n💡 Look at the false positives — why did the model flag them?")
print(f"   Often they contain words like 'free' or 'call' in a legitimate context.")
print(f"\n🎉 Exercise 3 complete! Error analysis is how real ML engineers improve models.")

## Summary: Your First Day

You explored real data, cleaned it, and built 3 progressively better baselines:

| Baseline | Approach | F1 Score |
|----------|----------|----------|
| Keywords | Rule-based matching | ~0.60 |
| Features | 8 hand-crafted features + LogReg | ~0.90 |
| TF-IDF | Word frequencies + LogReg | ~0.95 |

**Key takeaways:**
1. **Always start with EDA** — you found duplicates and class imbalance
2. **Simple baselines first** — they're fast and set a floor for comparison
3. **The right features matter more than the model** — TF-IDF + LogReg is surprisingly good
4. **Watch for data leakage** — split before preprocessing
5. **Error analysis > accuracy** — look at what the model gets wrong

**Next up**: Lesson 29 — Can a pre-trained Transformer beat TF-IDF? 